In [33]:
import re
import os
import pandas as pd
import numpy as np
from pathlib import Path

In [34]:
GTFS_DIR = Path().resolve().parents[2] / "data" / "belgique" / "mdb-686-202606040030"
OUTPUT_CSV = "../../../data/output/belgium_weekly_trains.csv"
DATA_SOURCE = "Belgique"

In [35]:
def load(filename, **kwargs):
    path = os.path.join(GTFS_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Fichier introuvable : {path}")
    return pd.read_csv(path, dtype=str, **kwargs)

_COUNTRY_SUFFIX = re.compile(r"\s*\(.*?\)")

def clean_city(name: str) -> str:
    if pd.isna(name):
        return name
    return _COUNTRY_SUFFIX.sub("", name).strip()

def build_active_dates(gtfs_dir: str) -> pd.DataFrame:
    cal_dates_path = os.path.join(gtfs_dir, "calendar_dates.txt")
    cal_path = os.path.join(gtfs_dir, "calendar.txt")
    rows = []

    if os.path.exists(cal_dates_path):
        cal_dates = pd.read_csv(cal_dates_path, dtype=str)
        active = cal_dates[cal_dates["exception_type"] == "1"].copy()
        active["date_dt"] = pd.to_datetime(active["date"], format="%Y%m%d")
        rows.append(active[["service_id", "date_dt"]])

    if os.path.exists(cal_path):
        cal = pd.read_csv(cal_path, dtype=str)
        day_cols = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]

        cal["start_dt"] = pd.to_datetime(cal["start_date"], format="%Y%m%d")
        cal["end_dt"] = pd.to_datetime(cal["end_date"],   format="%Y%m%d")
        cal[day_cols] = cal[day_cols].eq("1")

        chunks = []
        for _, row in cal.iterrows():
            active_days = [i for i, c in enumerate(day_cols) if row[c]]
            if not active_days:
                continue
            dates = pd.date_range(row["start_dt"], row["end_dt"], freq="D")
            dates = dates[dates.day_of_week.isin(active_days)]
            if len(dates):
                chunks.append(pd.DataFrame({"service_id": row["service_id"], "date_dt": dates}))

        if chunks:
            rows.append(pd.concat(chunks, ignore_index=True))

    if not rows:
        raise RuntimeError("Aucun fichier calendar_dates.txt ni calendar.txt trouvé.")

    active_all = pd.concat(rows, ignore_index=True).drop_duplicates()
    iso = active_all["date_dt"].dt.isocalendar()
    active_all["week_label"] = (
        iso["year"].astype(str) + "-W" + iso["week"].astype(str).str.zfill(2)
    )
    return active_all

In [36]:
def main():
    trips = load("trips.txt")
    routes = load("routes.txt")
    stops = load("stops.txt")
    stop_times = load("stop_times.txt")

    active = build_active_dates(GTFS_DIR)
    trips_active = trips.merge(active[["service_id", "week_label"]], on="service_id", how="inner")

    stop_times["stop_sequence"] = pd.to_numeric(stop_times["stop_sequence"], errors="coerce")
    st = stop_times[stop_times["trip_id"].isin(trips_active["trip_id"])].copy()

    idx_first = st.groupby("trip_id")["stop_sequence"].idxmin()
    idx_last = st.groupby("trip_id")["stop_sequence"].idxmax()

    first_stops = st.loc[idx_first, ["trip_id","stop_id"]].rename(columns={"stop_id":"origin_stop_id"})
    last_stops = st.loc[idx_last,  ["trip_id","stop_id"]].rename(columns={"stop_id":"destination_stop_id"})

    trips_od = (
        trips_active
        .merge(first_stops, on="trip_id", how="left")
        .merge(last_stops,  on="trip_id", how="left")
    )

    stop_names = stops[["stop_id","stop_name"]].drop_duplicates().copy()
    stop_names["city"] = (
        stop_names["stop_name"]
        .str.replace(r"\s*\(.*?\)", "", regex=True)
        .str.strip()
    )

    trips_od = (
        trips_od
        .merge(
            stop_names.rename(columns={"stop_id":"origin_stop_id","city":"origin_city"}),
            on="origin_stop_id", how="left"
        )
        .merge(
            stop_names.rename(columns={"stop_id":"destination_stop_id","city":"destination_city"}),
            on="destination_stop_id", how="left"
        )
    )

    routes_clean = routes.copy()
    split = routes_clean["route_long_name"].str.split("--", n=1, expand=True).apply(
        lambda col: col.str.strip()
    )
    routes_clean["origin_route"] = split[0]
    routes_clean["destination_route"] = split[1] if 1 in split.columns else None

    trips_od = trips_od.merge(
        routes_clean[["route_id","origin_route","destination_route"]],
        on="route_id", how="left"
    )
    trips_od["origin_city"] = trips_od["origin_city"].fillna(trips_od["origin_route"])
    trips_od["destination_city"] = trips_od["destination_city"].fillna(trips_od["destination_route"])

    od = trips_od[["origin_city","destination_city"]].to_numpy()
    trips_od["city_1"] = od.min(axis=1)
    trips_od["city_2"] = od.max(axis=1)

    weekly = (
        trips_od
        .groupby(["route_id","week_label","city_1","city_2"], dropna=False)
        .agg(weekly_train=("trip_id","count"))
        .reset_index()
        .rename(columns={"city_1":"origin_city","city_2":"destination_city"})
    )


    weekly["desserte_type"] = np.select(
        [weekly["weekly_train"] < 7, weekly["weekly_train"] <= 56],
        ["Sous-desservi", "Desserte Normale"],
        default="Bien desservi"
    )

    weekly.insert(0, "data_source", DATA_SOURCE)

    # prendre ça pour la db
    output = (
        weekly[["data_source","route_id","origin_city","destination_city",
                "weekly_train","desserte_type"]]
        .sort_values(["route_id","origin_city"])
        .reset_index(drop=True)
    )

    out_path = os.path.join(GTFS_DIR, OUTPUT_CSV)
    output.to_csv(out_path, index=False, encoding="utf-8-sig")

In [37]:
if __name__ == "__main__":
    main()